# ingest_weather

Pulls public historical weather (Open-Meteo) for the 15 Contoso store cities into `bronze/Tables/dbo/weather`.

Incremental + idempotent:
- First run: backfills from 90 days ago through yesterday (matches the seed window).
- Subsequent runs: pulls from the day AFTER the latest row already in the table through yesterday.
- Re-runs in the same day are a no-op (already up to date).
- Uses Delta MERGE on (date, store_id) so overlapping windows just refresh in place.

In [ ]:
# Parameters baked by deploy.ps1 / the orchestration pipeline.
workspace_id = ""
lakehouse_id = ""
table_name   = "weather"

# Maximum backfill window if the table is empty (first run after deploy).
# Matches seed/00_seed_historical_data's 90-day fiscal-quarter window.
default_backfill_days = 90

In [ ]:
# Store list mirrors seed/00_seed_historical_data STORES.
STORES = [
    (1,  "New York",      "NY", 40.7831,  -73.9712),
    (2,  "Boston",        "MA", 42.3601,  -71.0589),
    (3,  "Philadelphia",  "PA", 39.9526,  -75.1652),
    (4,  "Atlanta",       "GA", 33.7490,  -84.3880),
    (5,  "Miami",         "FL", 25.7617,  -80.1918),
    (6,  "Nashville",     "TN", 36.1627,  -86.7816),
    (7,  "Dallas",        "TX", 32.7767,  -96.7970),
    (8,  "Austin",        "TX", 30.2672,  -97.7431),
    (9,  "Chicago",       "IL", 41.8781,  -87.6298),
    (10, "Minneapolis",   "MN", 44.9778,  -93.2650),
    (11, "Denver",        "CO", 39.7392, -104.9903),
    (12, "Phoenix",       "AZ", 33.4484, -112.0740),
    (13, "Los Angeles",   "CA", 34.0522, -118.2437),
    (14, "Seattle",       "WA", 47.6062, -122.3321),
    (15, "Portland",      "OR", 45.5152, -122.6784),
]
print(f"{len(STORES)} stores")

In [ ]:
# Resolve the target Delta path and figure out which date window to fetch.
# - If the table doesn't exist yet, or is empty, backfill default_backfill_days.
# - Otherwise, fetch from max(date) + 1 through yesterday.
# - If start > end, there's nothing to do (already current).
import datetime as _dt
from delta.tables import DeltaTable
from pyspark.sql import functions as F

if not (workspace_id and lakehouse_id):
    raise ValueError("workspace_id and lakehouse_id must be supplied (deploy.ps1 bakes them).")

table_path = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables/dbo/{table_name}"
end_date   = _dt.date.today() - _dt.timedelta(days=1)

if DeltaTable.isDeltaTable(spark, table_path):
    existing = spark.read.format("delta").load(table_path)
    latest_row = existing.agg(F.max("date").alias("d")).collect()[0]
    latest = latest_row["d"]
    if latest is None:
        start_date = end_date - _dt.timedelta(days=default_backfill_days - 1)
        mode = "backfill (table empty)"
    else:
        # Coerce in case an older deploy stored date as StringType.
        if isinstance(latest, str):
            latest = _dt.date.fromisoformat(latest)
        start_date = latest + _dt.timedelta(days=1)
        mode = f"incremental (latest={latest})"
else:
    start_date = end_date - _dt.timedelta(days=default_backfill_days - 1)
    mode = "backfill (no table)"

print(f"mode: {mode}")
print(f"window: {start_date} -> {end_date}")

if start_date > end_date:
    print("weather table is already current; exiting.")
    try:
        notebookutils.notebook.exit("up-to-date")
    except NameError:
        pass

In [ ]:
import requests, time

WMO_DESC = {
    0:'Clear', 1:'Mainly clear', 2:'Partly cloudy', 3:'Overcast',
    45:'Fog', 48:'Depositing rime fog',
    51:'Light drizzle', 53:'Moderate drizzle', 55:'Dense drizzle',
    56:'Light freezing drizzle', 57:'Dense freezing drizzle',
    61:'Slight rain', 63:'Moderate rain', 65:'Heavy rain',
    66:'Light freezing rain', 67:'Heavy freezing rain',
    71:'Slight snow', 73:'Moderate snow', 75:'Heavy snow', 77:'Snow grains',
    80:'Slight rain showers', 81:'Moderate rain showers', 82:'Violent rain showers',
    85:'Slight snow showers', 86:'Heavy snow showers',
    95:'Thunderstorm', 96:'Thunderstorm w/ slight hail', 99:'Thunderstorm w/ heavy hail',
}

API = 'https://historical-forecast-api.open-meteo.com/v1/forecast'
DAILY = 'temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum,wind_speed_10m_max,weather_code'

def _f(x): return None if x is None else float(x)
def _i(x): return None if x is None else int(x)

def _fetch(lat, lon):
    last = None
    for attempt in range(4):
        try:
            r = requests.get(API, params={'latitude':lat,'longitude':lon,'start_date':start_date.isoformat(),'end_date':end_date.isoformat(),'daily':DAILY,'timezone':'America/New_York'}, timeout=90)
            r.raise_for_status()
            return r.json()['daily']
        except Exception as e:
            last = e
            time.sleep(2 ** attempt)
    raise last

rows = []
for sid, city, state, lat, lon in STORES:
    d = _fetch(lat, lon)
    for i in range(len(d['time'])):
        wc = d['weather_code'][i]
        tmax = _f(d['temperature_2m_max'][i])
        tmin = _f(d['temperature_2m_min'][i])
        rows.append({
            'date': d['time'][i],
            'store_id': int(sid),
            'city': city,
            'state': state,
            'latitude': float(lat),
            'longitude': float(lon),
            'temperature_max_c': tmax,
            'temperature_min_c': tmin,
            'temperature_max_f': None if tmax is None else round(tmax*9/5+32, 1),
            'temperature_min_f': None if tmin is None else round(tmin*9/5+32, 1),
            'precipitation_mm':  _f(d['precipitation_sum'][i]),
            'snowfall_cm':       _f(d['snowfall_sum'][i]),
            'wind_speed_max_kmh':_f(d['wind_speed_10m_max'][i]),
            'weather_code':      _i(wc),
            'weather_description': WMO_DESC.get(wc, f'Unknown ({wc})'),
        })
    time.sleep(0.05)
print(f'fetched {len(rows)} rows')

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

schema = StructType([
    StructField('date',                 StringType(),  False),
    StructField('store_id',             IntegerType(), False),
    StructField('city',                 StringType(),  False),
    StructField('state',                StringType(),  False),
    StructField('latitude',             DoubleType(),  False),
    StructField('longitude',            DoubleType(),  False),
    StructField('temperature_max_c',    DoubleType(),  True),
    StructField('temperature_min_c',    DoubleType(),  True),
    StructField('temperature_max_f',    DoubleType(),  True),
    StructField('temperature_min_f',    DoubleType(),  True),
    StructField('precipitation_mm',     DoubleType(),  True),
    StructField('snowfall_cm',          DoubleType(),  True),
    StructField('wind_speed_max_kmh',   DoubleType(),  True),
    StructField('weather_code',         IntegerType(), True),
    StructField('weather_description',  StringType(),  True),
])

_cols = [f.name for f in schema.fields]
_tuples = [tuple(r.get(c) for c in _cols) for r in rows]
df = spark.createDataFrame(_tuples, schema=schema).withColumn('date', F.to_date('date'))

if DeltaTable.isDeltaTable(spark, table_path):
    # Idempotent merge on (date, store_id). Handles overlapping windows + reruns.
    tgt = DeltaTable.forPath(spark, table_path)
    (tgt.alias('t')
        .merge(df.alias('s'), 't.date = s.date AND t.store_id = s.store_id')
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    print(f'merged {df.count()} rows into {table_path}')
else:
    df.write.format('delta').mode('overwrite').save(table_path)
    print(f'wrote {df.count()} rows -> {table_path} (initial)')

In [ ]:
out = spark.read.format('delta').load(table_path)
print(f'total rows now: {out.count()}')
out.agg(F.min('date').alias('min_date'), F.max('date').alias('max_date')).show(truncate=False)
out.orderBy(F.desc('date'), 'store_id').show(10, truncate=False)